In [ ]:
!pip install matplotlib

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt  # for the heatmap


In [2]:
PARQUET_PATH = r"D:\Master\IBD\traffic_full.parquet"  # change if your path is different

df = pd.read_parquet(PARQUET_PATH)

# Remove the dask index if it exists
if "__null_dask_index__" in df.columns:
    df = df.drop(columns=["__null_dask_index__"])

df.head()


,collision_index,latitude,longitude,speed_limit,weather_conditions,road_surface_conditions,light_conditions,collision_severity,vehicle_reference,vehicle_type,...,sex_of_driver,engine_capacity_cc,propulsion_code,journey_purpose_of_driver,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,casualty_severity,lsoa_of_casualty
__null_dask_index__,,,,,,,,,,,,,,,,,,,,,
0,200701CW10148,51.533029,-0.188564,30.0,2.0,2.0,1.0,2.0,1.0,19.0,...,1.0,2188.0,2.0,1.0,1.0,1.0,2.0,60.0,2.0,-1
1,200701CW10148,51.533029,-0.188564,30.0,2.0,2.0,1.0,2.0,2.0,9.0,...,2.0,1332.0,1.0,-1.0,1.0,1.0,2.0,60.0,2.0,-1
2,2011440140021,50.902997,-1.335113,40.0,1.0,1.0,1.0,3.0,1.0,9.0,...,1.0,1590.0,1.0,2.0,1.0,1.0,1.0,20.0,3.0,E01022790
3,2011440140021,50.902997,-1.335113,40.0,1.0,1.0,1.0,3.0,2.0,9.0,...,1.0,1242.0,1.0,2.0,1.0,1.0,1.0,20.0,3.0,E01022790
4,2015100633715,54.931762,-1.485336,60.0,1.0,1.0,1.0,2.0,2.0,19.0,...,1.0,2198.0,2.0,2.0,1.0,1.0,1.0,29.0,2.0,E01008694


In [3]:
numeric_features = [
    "latitude", "longitude",
    "speed_limit",
    "age_of_vehicle", "age_of_driver", "engine_capacity_cc",
    "age_of_casualty",
    # you can add more if you want and they are numeric
]

# Convert needed columns to numeric (in case they are strings)
for col in numeric_features + ["casualty_severity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep only rows where severity is known
df = df.dropna(subset=["casualty_severity"]).copy()
df["casualty_severity"] = df["casualty_severity"].astype(int)

# Binary target: 1 if severity is 1 or 2, else 0
df["is_serious"] = df["casualty_severity"].isin([1, 2]).astype(int)

df[["casualty_severity", "is_serious"]].head()


,casualty_severity,is_serious
__null_dask_index__,,
0,2,1
1,2,1
2,3,0
3,3,0
4,2,1


In [4]:
cols_for_corr = numeric_features + ["casualty_severity", "is_serious"]

corr_matrix = df[cols_for_corr].corr(method="pearson")
corr_matrix


,latitude,longitude,speed_limit,age_of_vehicle,age_of_driver,engine_capacity_cc,age_of_casualty,casualty_severity,is_serious
latitude,1.000000,-0.419581,0.041809,-0.039587,0.040208,0.019510,0.018626,-0.026197,0.024763
longitude,-0.419581,1.000000,-0.054099,0.008334,-0.061280,-0.003821,-0.026014,0.008693,-0.005986
speed_limit,0.041809,-0.054099,1.000000,0.001403,0.106670,0.087445,0.066281,-0.068668,0.056643
age_of_vehicle,-0.039587,0.008334,0.001403,1.000000,0.076710,0.253688,0.011953,-0.006496,0.005228
age_of_driver,0.040208,-0.061280,0.106670,0.076710,1.000000,0.138998,0.349973,-0.038186,0.035431
engine_capacity_cc,0.019510,-0.003821,0.087445,0.253688,0.138998,1.000000,0.055664,-0.006362,-0.001866
age_of_casualty,0.018626,-0.026014,0.066281,0.011953,0.349973,0.055664,1.000000,-0.059954,0.053801
casualty_severity,-0.026197,0.008693,-0.068668,-0.006496,-0.038186,-0.006362,-0.059954,1.000000,-0.965375
is_serious,0.024763,-0.005986,0.056643,0.005228,0.035431,-0.001866,0.053801,-0.965375,1.000000


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)

plt.title("Correlation matrix – numeric features vs severity")
plt.tight_layout()
plt.show()


In [5]:
def seriousness_table(col, min_count=1000):
    """
    For a categorical column `col`, show:
    - number of accidents
    - % serious/fatal (is_serious = 1)
    - % fatal (casualty_severity == 1)
    """
    grouped = df.groupby(col).agg(
        n_accidents=("is_serious", "size"),
        serious_rate=("is_serious", "mean"),
        fatal_rate=("casualty_severity", lambda s: (s == 1).mean())
    )
    # keep only categories with enough data
    grouped = grouped[grouped["n_accidents"] >= min_count]
    grouped["serious_rate_%"] = (grouped["serious_rate"] * 100).round(2)
    grouped["fatal_rate_%"] = (grouped["fatal_rate"] * 100).round(2)
    return grouped[["n_accidents", "serious_rate_%", "fatal_rate_%"]].sort_values(
        "serious_rate_%", ascending=False
    )


In [9]:
def seriousness_table(col, min_count=1000):
    """
    For a column `col`, show:
    - number of accidents
    - % serious/fatal (is_serious = 1)
    - % fatal (casualty_severity == 1)

    min_count = minimum number of accidents to keep that category.
    """
    grouped = df.groupby(col).agg(
        n_accidents=("is_serious", "size"),
        serious_rate=("is_serious", "mean"),
        fatal_rate=("casualty_severity", lambda s: (s == 1).mean())
    )
    grouped = grouped[grouped["n_accidents"] >= min_count]
    grouped["serious_rate_%"] = (grouped["serious_rate"] * 100).round(2)
    grouped["fatal_rate_%"] = (grouped["fatal_rate"] * 100).round(2)
    return grouped[["n_accidents", "serious_rate_%", "fatal_rate_%"]].sort_values(
        "serious_rate_%", ascending=False
    )


In [11]:
cat_columns = [
    "speed_limit",              # numeric but treated as categories (20,30,...)
    "weather_conditions",
    "road_surface_conditions",
    "light_conditions",
    "vehicle_type",
    "sex_of_driver",
    "propulsion_code",
    "journey_purpose_of_driver",
    "casualty_class",
    "sex_of_casualty",
    "lsoa_of_casualty",         # MANY categories – will still work but big table
]


In [12]:
tables = {}

for col in cat_columns:
    print(f"Building table for {col}...")
    # For LSOA we might want a higher min_count to avoid tens of thousands of rows
    if col == "lsoa_of_casualty":
        tables[col] = seriousness_table(col, min_count=5000)
    else:
        tables[col] = seriousness_table(col, min_count=1000)


Building table for speed_limit...
Building table for weather_conditions...
Building table for road_surface_conditions...
Building table for light_conditions...
Building table for vehicle_type...
Building table for sex_of_driver...
Building table for propulsion_code...
Building table for journey_purpose_of_driver...
Building table for casualty_class...
Building table for sex_of_casualty...
Building table for lsoa_of_casualty...


In [16]:
with pd.ExcelWriter("severity_correlations_tables.xlsx") as writer:
    for col, df_table in tables.items():
        # Excel sheet names can't be longer than 31 chars
        sheet_name = col[:31]
        df_table.to_excel(writer, sheet_name=sheet_name)
